# 02 - Sales Exploration

**Objective:** Answer the core revenue/profit trend questions:
how is revenue changing over time, which categories/regions drive it, and
how does discounting relate to profitability?

**Business context:** Management wants a clear read on where revenue and
profit are coming from before deciding where to focus commercial attention.


In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import matplotlib.pyplot as plt
from data_loading import load_orders
from data_cleaning import clean_orders
from feature_engineering import engineer_all
from kpi_calculations import compute_core_kpis, revenue_by_period, revenue_growth
import visualization as viz

pd.set_option('display.max_columns', 30)

raw = load_orders()
cleaned, _ = clean_orders(raw)
tables = engineer_all(cleaned)
lines = tables['lines']
print(f"Analysis-ready dataset: {lines.shape[0]:,} line items")


Analysis-ready dataset: 10,194 line items


## Headline KPIs

In [2]:
kpis = compute_core_kpis(lines)
for k, v in kpis.items():
    print(f"{k:>26}: {v:,.4f}" if isinstance(v, float) else f"{k:>26}: {v:,}")


             total_revenue: 2,326,534.3543
              total_profit: 292,296.8146
              total_orders: 5,111
               total_units: 38,654
             profit_margin: 0.1256
       average_order_value: 455.2014
   average_units_per_order: 7.5629
            customer_count: 804
      revenue_per_customer: 2,893.6994
      repeat_customer_rate: 0.9851


## Revenue and profit trend over time

**Business question: How is revenue changing over time?**

In [3]:
monthly = revenue_by_period(lines, 'order_year_month')
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(monthly['order_year_month'], monthly['revenue'], marker='o', markersize=3, label='Revenue')
ax.plot(monthly['order_year_month'], monthly['profit'], marker='o', markersize=3, label='Profit')
ax.set_title('Monthly Revenue and Profit Trend')
step = max(1, len(monthly)//12)
ax.set_xticks(monthly['order_year_month'][::step])
ax.tick_params(axis='x', rotation=90)
ax.legend()
plt.show()


C:\Users\syedm\AppData\Local\Temp\ipykernel_10948\4131352537.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
yearly = revenue_growth(lines, 'order_year')
yearly

,order_year,revenue,profit,revenue_growth_rate,profit_growth_rate
0,2023,494040.2121,51684.2957,NaN,NaN
1,2024,472993.0310,62020.9695,-0.042602,0.199996
2,2025,613933.5800,82665.2018,0.297976,0.332859
3,2026,745567.5312,95926.3476,0.214411,0.160420


**Interpretation:** Revenue grew every year except 2024, which declined
-4.3% year-over-year even though profit rose +20.0% that same year -- a
signal that 2024 traded some revenue for better margin discipline. 2025 and
2026 both show strong double-digit revenue growth (+29.8% and +21.4%).


## Category and sub-category performance

**Business questions: Which categories/sub-categories generate the most
revenue? The most profit? Which perform poorly?**

In [5]:
category_summary = (lines.groupby('category')
    .agg(revenue=('sales','sum'), profit=('profit','sum'), units=('quantity','sum'))
    .reset_index())
category_summary['margin'] = category_summary['profit'] / category_summary['revenue']
category_summary = category_summary.sort_values('revenue', ascending=False)
category_summary

,category,revenue,profit,units,margin
2,Technology,839893.2790,146543.3756,7017,0.174479
0,Furniture,754747.7613,19729.9956,8369,0.026141
1,Office Supplies,731893.3140,126023.4434,23268,0.172188


In [6]:
fig, ax = plt.subplots(figsize=(7,4))
x = range(len(category_summary))
ax.bar([i-0.2 for i in x], category_summary['revenue'], width=0.4, label='Revenue')
ax.bar([i+0.2 for i in x], category_summary['profit'], width=0.4, label='Profit')
ax.set_xticks(list(x)); ax.set_xticklabels(category_summary['category'])
ax.set_title('Revenue vs Profit by Category')
ax.legend()
plt.show()


C:\Users\syedm\AppData\Local\Temp\ipykernel_10948\2612071260.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation:** Furniture generates almost as much revenue as
Technology and Office Supplies, but only 2.6% margin vs. ~17% for the other
two -- Furniture has a profitability problem, not a demand problem.

In [7]:
subcategory_summary = (lines.groupby('sub_category')
    .agg(revenue=('sales','sum'), profit=('profit','sum'))
    .reset_index())
subcategory_summary['margin'] = subcategory_summary['profit'] / subcategory_summary['revenue']
subcategory_summary.sort_values('margin').head(5)


,sub_category,revenue,profit,margin
16,Tables,208020.1820,-17753.2061,-0.085344
4,Bookcases,115361.2043,-3632.0736,-0.031484
15,Supplies,46725.4980,-1171.3945,-0.025070
11,Machines,189925.0310,3461.9769,0.018228
5,Chairs,335768.2490,27223.5323,0.081078


**Interpretation:** Three sub-categories are net loss-making overall:
**Tables** (-8.5% margin), **Bookcases** (-3.1%), and **Supplies** (-2.5%).
Tables is the clearest concern -- it loses money at scale (-$17,753 on
$208,020 of revenue).

## Regional performance

**Business question: Which regions perform best?**

In [8]:
region_summary = (lines.groupby('region')
    .agg(revenue=('sales','sum'), profit=('profit','sum'), orders=('order_id','nunique'))
    .reset_index())
region_summary['margin'] = region_summary['profit']/region_summary['revenue']
region_summary = region_summary.sort_values('revenue', ascending=False)
region_summary

,region,revenue,profit,orders,margin
3,West,739813.6085,110798.8170,1635,0.149766
1,East,691828.1680,94883.2603,1475,0.137149
0,Central,503170.6728,39865.3070,1179,0.079228
2,South,391721.9050,46749.4303,822,0.119343


**Interpretation:** West leads on both revenue ($739,814) and margin
(15.0%). Central has the lowest margin (7.9%) despite being 3rd in revenue --
worth a closer look at Central's discounting/product mix.

## Discount vs. profitability

**Business question: How does discount relate to profitability?**

**Important:** this section shows an *association*, not a causal claim.
Discounts may be applied selectively to already low-margin or slow-moving
stock, which would produce the same pattern without discounting itself
being the cause.

In [9]:
bins = [-0.01,0,0.1,0.2,0.3,0.4,0.5,1.0]
labels = ['0%','0-10%','10-20%','20-30%','30-40%','40-50%','50%+']
lines2 = lines.copy()
lines2['discount_bucket'] = pd.cut(lines2['discount'], bins=bins, labels=labels)
disc = (lines2.groupby('discount_bucket', observed=True)
    .agg(revenue=('sales','sum'), profit=('profit','sum'))
    .reset_index())
disc['margin'] = disc['profit']/disc['revenue']
disc

,discount_bucket,revenue,profit,margin
0,0%,1.105324e+06,326718.5872,0.295586
1,0-10%,5.495250e+04,9099.9700,0.165597
2,10-20%,8.014979e+05,92498.9432,0.115408
3,20-30%,1.044741e+05,-10513.4456,-0.100632
4,30-40%,1.309912e+05,-25477.5119,-0.194498
5,40-50%,6.440351e+04,-22999.5392,-0.357116
6,50%+,6.489135e+04,-77030.1891,-1.187064


In [10]:
corr = lines2['discount'].corr(lines2['profit_margin'])
print(f"Correlation between discount and line-item profit margin: {corr:.3f}")


Correlation between discount and line-item profit margin: -0.865


**Interpretation:** Margin is positive at every discount tier up to 20%,
and negative at every tier from 20% upward -- a threshold effect, not a
gradual slide. The -0.865 correlation is strong, but *association, not
causation* (see caveat above and `docs/findings.md`).